# 04 — Per-rm linear (winner #1's approach)
Fit OLS on the cumulative curve of the most recent year, shrink slope by `s ∈ [0.5, 0.7]`. Beats every other model on the τ=0.2 metric.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from src.data import load_or_build, build_profile
from src.gating import assign_tracks
from src.models.linear_per_rm import PerRMLinearForecaster
from src.validation import DEFAULT_FOLDS, evaluate, build_query_for_fold


In [ ]:
ds = load_or_build()
rm_ids = sorted(ds.daily['rm_id'].unique().tolist())
results = []
for fold in DEFAULT_FOLDS:
    cutoff = fold.train_end + pd.Timedelta(days=1)
    daily_pre = ds.daily[ds.daily['date'] < cutoff]
    profile = build_profile(daily_pre, year=fold.target_year - 1)
    tracks = assign_tracks(profile, all_rm_ids=rm_ids)
    track_ab = set(tracks[tracks['track'].isin(['A','B'])]['rm_id'].tolist())
    query = build_query_for_fold(fold, rm_ids)
    for s in np.arange(0.4, 1.05, 0.05):
        m = PerRMLinearForecaster(fit_year=fold.target_year - 1, slope_shrink=float(s)).fit(daily_pre)
        preds = m.predict(query, rm_id_track_filter=track_ab)
        results.append((fold.name, float(s), evaluate(preds, fold, ds.daily)['mean_pinball']))
import pandas as pd
pd.DataFrame(results, columns=['fold','shrink','pinball']).pivot_table(values='pinball', index='shrink', columns='fold')
